# Лабораторная работа №3 — Оптимизация гиперпараметров

**Задача:** регрессия для предсказания `quality` вина из лабораторной №1.  
**Модель:** `GradientBoostingRegressor`.  
**Цель:** сравнить Grid Search и Random Search, исследовать влияние гиперпараметров и оценить финальную модель на отложенной тестовой выборке.

Тестовая выборка используется только один раз — после выбора параметров по кросс-валидации на обучающих данных.


## 1. Импорт библиотек и настройки

Используем RMSE: метрика выражена в единицах `quality`, а крупные ошибки штрафуются сильнее. Для CV в `scikit-learn` удобен scorer `neg_root_mean_squared_error`; перед выводом меняем знак, чтобы получить привычный RMSE, где меньше — лучше.


In [ ]:
import io
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split, KFold, GridSearchCV, RandomizedSearchCV,
    cross_val_score
)
from sklearn.metrics import mean_squared_error
from IPython.display import display

RANDOM_STATE = 42
CV_SPLITS = 5
SCORING = "neg_root_mean_squared_error"
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Загрузка и подготовка данных

Используем объединённый датасет красных и белых вин, как в лабораторной №1. Тип вина преобразуется в бинарный признак. Сначала отделяем тестовую часть, после этого все решения о гиперпараметрах принимаются только на `X_train`, `y_train`.


In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"

def load_wine_quality():
    red_path = Path("/mnt/data/winequality-red.csv")
    white_path = Path("/mnt/data/winequality-white.csv")
    if not red_path.exists():
        red_path = Path("winequality-red.csv")
    if not white_path.exists():
        white_path = Path("winequality-white.csv")
    if red_path.exists() and white_path.exists():
        red = pd.read_csv(red_path, sep=";")
        white = pd.read_csv(white_path, sep=";")
    else:
        with urllib.request.urlopen(UCI_URL, timeout=30) as response:
            archive_bytes = response.read()
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as zf:
            red_name = next(n for n in zf.namelist() if n.endswith("winequality-red.csv"))
            white_name = next(n for n in zf.namelist() if n.endswith("winequality-white.csv"))
            red = pd.read_csv(zf.open(red_name), sep=";")
            white = pd.read_csv(zf.open(white_name), sep=";")
    red["wine_type"] = "red"
    white["wine_type"] = "white"
    return pd.concat([red, white], ignore_index=True)

df = load_wine_quality()
model_df = df.copy()
model_df["wine_type"] = model_df["wine_type"].map({"red": 0, "white": 1}).astype(int)
X = model_df.drop(columns="quality")
y = model_df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f"Всего объектов: {len(df)}; train: {len(X_train)}; test: {len(X_test)}")
print("Пропуски в данных:", int(model_df.isna().sum().sum()))


## 3. Baseline до оптимизации

Исходная модель — `GradientBoostingRegressor` с параметрами из лабораторной №1: `n_estimators=300`, `learning_rate=0.04`, `max_depth=2`. Это baseline. Его CV-оценка вычисляется на пяти folds только обучающей выборки. Тестовая метрика пока не рассчитывается.

**Зачем нужна кросс-валидация?** Она оценивает качество на нескольких разбиениях и снижает зависимость вывода от одного случайного разбиения. Одного train/validation разбиения недостаточно: результат может сильно зависеть от того, какие объекты попали в validation. Если многократно выбирать параметры по test, мы фактически подстраиваемся под тестовые данные, и итоговая оценка теряет независимость.

Выбрано 5 folds: это компромисс между устойчивостью оценки и вычислительными затратами. На каждом шаге четыре части используются для обучения, одна — для проверки.


In [ ]:
cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
baseline_params = {
    "n_estimators": 300,
    "learning_rate": 0.04,
    "max_depth": 2,
    "random_state": RANDOM_STATE,
}
baseline_model = GradientBoostingRegressor(**baseline_params)
baseline_cv_scores = -cross_val_score(
    baseline_model, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
)
print("Параметры baseline:", baseline_params)
print("RMSE по folds:", np.round(baseline_cv_scores, 4))
print(f"Средний CV RMSE: {baseline_cv_scores.mean():.4f} ± {baseline_cv_scores.std():.4f}")


## 4. Гиперпараметры и ожидания

Подбираются три параметра, влияющие на сложность и качество бустинга:

- `n_estimators` — число последовательно добавляемых деревьев. Больше деревьев могут улучшить аппроксимацию, но увеличивают время и при прочих равных могут усилить переобучение.
- `max_depth` — максимальная глубина каждого дерева. Маленькая глубина ограничивает сложность и может привести к недообучению; большая позволяет описывать взаимодействия, но повышает риск переобучения.
- `min_samples_leaf` — минимальное число объектов в листе. Увеличение сглаживает предсказания и ограничивает сложность; слишком большое значение может привести к недообучению.

Значения подобраны как небольшой разумный диапазон вокруг исходной конфигурации. `learning_rate` оставлен фиксированным на исходном значении 0.04, чтобы анализировать влияние трёх выбранных параметров при ограниченном бюджете.


## 5. Grid Search

Сетка содержит 2 × 3 × 3 = **18 комбинаций**. Каждая комбинация оценивается на пяти folds, всего выполняется 90 обучений. Оптимизируем средний RMSE.


In [ ]:
param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [1, 2, 3],
    "min_samples_leaf": [1, 2, 4],
}
grid_combinations = int(np.prod([len(v) for v in param_grid.values()]))

grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(
        learning_rate=0.04, random_state=RANDOM_STATE
    ),
    param_grid=param_grid,
    scoring=SCORING,
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
start = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_time = time.perf_counter() - start

grid_best_rmse = -grid_search.best_score_
print("Пространство поиска:", param_grid)
print("Проверено комбинаций:", grid_combinations)
print("Лучшие параметры:", grid_search.best_params_)
print(f"Лучший средний CV RMSE: {grid_best_rmse:.4f}")
print(f"Время поиска: {grid_time:.2f} с")


## 6. Random Search с сопоставимым бюджетом

Random Search проверяет **18 случайных комбинаций** из тех же диапазонов, что и Grid Search, то есть столько же комбинаций и folds. В этом дискретном пространстве случайный поиск может повторить отдельную комбинацию; фактическое количество уникальных параметров выводится отдельно. При большом числе параметров и непрерывных диапазонах Random Search часто эффективнее: он распределяет фиксированный бюджет по всему пространству, а не тратит его на плотный перебор каждого параметра.


In [ ]:
random_search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(
        learning_rate=0.04, random_state=RANDOM_STATE
    ),
    param_distributions=param_grid,
    n_iter=grid_combinations,
    scoring=SCORING,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True,
)
start = time.perf_counter()
random_search.fit(X_train, y_train)
random_time = time.perf_counter() - start

random_best_rmse = -random_search.best_score_
random_unique = random_search.cv_results_["params"]
random_unique_count = len({tuple(sorted(p.items())) for p in random_unique})
print("Запрошено комбинаций:", grid_combinations)
print("Уникальных проверенных комбинаций:", random_unique_count)
print("Лучшие параметры:", random_search.best_params_)
print(f"Лучший средний CV RMSE: {random_best_rmse:.4f}")
print(f"Время поиска: {random_time:.2f} с")


## 7. Сравнение стратегий поиска и влияние гиперпараметров

Сравниваем лучшие CV RMSE и время. Для анализа влияния каждого параметра агрегируем CV результаты Grid Search по остальным параметрам. Это усреднение по небольшой заданной сетке, поэтому оно показывает тенденцию в исследованной области, а не универсальную зависимость.


In [ ]:
search_comparison = pd.DataFrame([
    {"Метод": "Grid Search", "Проверено комбинаций": grid_combinations,
     "Лучший CV RMSE": grid_best_rmse, "Время, с": grid_time},
    {"Метод": "Random Search", "Проверено комбинаций": grid_combinations,
     "Лучший CV RMSE": random_best_rmse, "Время, с": random_time},
])
display(search_comparison.round(4))

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results["mean_rmse"] = -cv_results["mean_test_score"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, param in zip(axes, ["n_estimators", "max_depth", "min_samples_leaf"]):
    summary = cv_results.groupby(f"param_{param}")["mean_rmse"].agg(["mean", "std"])
    ax.errorbar(summary.index.astype(float), summary["mean"], yerr=summary["std"], marker="o", capsize=4)
    ax.set_title(f"Влияние {param}")
    ax.set_xlabel(param)
    ax.set_ylabel("Средний CV RMSE (меньше — лучше)")
plt.tight_layout()
plt.show()


На графиках меньший RMSE соответствует лучшему результату. Высокая ошибка у малых `n_estimators` или `max_depth` может указывать на недообучение; рост ошибки при более сложных конфигурациях и особенно большой разрыв между train и validation score может указывать на переобучение. При интерпретации учитывайте, что здесь показано усреднение по прочим значениям параметров.


## 8. Устойчивость по folds

Сравним отдельные fold scores baseline, Grid Search и Random Search. Значения параметров поиска были выбраны на тех же CV folds, поэтому оценки лучших моделей слегка оптимистичны как оценки процедуры подбора; отдельный тест остаётся независимым. Малый разброс между folds означает более стабильное качество на данных с разным составом.


In [ ]:
def fold_rmse(estimator):
    return -cross_val_score(
        estimator, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
    )

grid_fold_scores = fold_rmse(grid_search.best_estimator_)
random_fold_scores = fold_rmse(random_search.best_estimator_)
fold_table = pd.DataFrame({
    "Fold": np.arange(1, CV_SPLITS + 1),
    "Baseline": baseline_cv_scores,
    "Grid Search": grid_fold_scores,
    "Random Search": random_fold_scores,
})
display(fold_table.round(4))
fold_summary = fold_table.drop(columns="Fold").agg(["mean", "std"]).T.rename(
    columns={"mean": "Средний RMSE", "std": "Стандартное отклонение"}
)
display(fold_summary.round(4))
fold_table.set_index("Fold").plot(marker="o", figsize=(9, 4))
plt.ylabel("RMSE на fold")
plt.title("Устойчивость качества между folds")
plt.xticks(range(1, CV_SPLITS + 1))
plt.tight_layout()
plt.show()


Две модели с одинаковым средним RMSE могут различаться по устойчивости: одна может одинаково работать на всех folds, а другая — хорошо на части и заметно хуже на остальных. Это может отражать чувствительность к составу обучающих данных или неоднородность выборки.


## 9. Финальная оценка на test

После завершения выбора параметров оцениваем исходную модель и обе лучшие модели на тесте. До этого шага test не участвовал ни в подборе, ни в сравнении конфигураций.


In [ ]:
models = {
    "Baseline": baseline_model,
    "Grid Search": grid_search.best_estimator_,
    "Random Search": random_search.best_estimator_,
}
final_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    test_rmse = mean_squared_error(y_test, model.predict(X_test)) ** 0.5
    if name == "Baseline":
        cv_rmse = baseline_cv_scores.mean()
        cv_std = baseline_cv_scores.std()
    elif name == "Grid Search":
        cv_rmse = grid_best_rmse
        cv_std = grid_fold_scores.std()
    else:
        cv_rmse = random_best_rmse
        cv_std = random_fold_scores.std()
    final_rows.append({
        "Модель": name,
        "Параметры": baseline_params if name == "Baseline" else model.get_params(),
        "CV RMSE": cv_rmse,
        "Разброс CV (std)": cv_std,
        "Test RMSE": test_rmse,
    })
final_results = pd.DataFrame(final_rows).set_index("Модель")
display(final_results[["CV RMSE", "Разброс CV (std)", "Test RMSE"]].round(4))
print("Выбранные параметры Grid Search:", grid_search.best_params_)
print("Выбранные параметры Random Search:", random_search.best_params_)


## 10. Итоговые выводы

Заполните выводы по фактическим результатам таблиц выше после запуска ноутбука:

1. **Качество подбора.** Сравните лучшие CV RMSE Grid Search и Random Search с baseline. Укажите, насколько снизился RMSE относительно baseline (в абсолютных единицах и процентах).
2. **Сопоставление CV и test.** Сравните ранжирование моделей по CV и test. Если улучшение на CV не подтвердилось на test, возможные причины — случайность разбиения, оптимизм из-за выбора лучшего результата среди многих комбинаций и ограниченный размер test.
3. **Поиск.** Укажите, какая стратегия дала лучшее качество при одинаковом числе комбинаций, и сравните время. Random Search особенно полезен при большом или непрерывном пространстве параметров и ограниченном бюджете.
4. **Сложность.** По графикам опишите, где видны признаки недообучения, разумной сложности и возможного переобучения. Если явного роста ошибки на исследованной сетке нет, так и отметьте: диапазон эксперимента мог не захватить переобучение.
5. **Устойчивость.** Сравните стандартные отклонения RMSE на folds. Модель с меньшим разбросом менее чувствительна к конкретному разбиению в этом эксперименте.

Все выводы о качестве относятся к выбранному разбиению и фиксированному seed; тестовая выборка использована только для заключительной проверки.
